In [0]:
# Filename: src/dlt_pipelines/streaming_pipeline.py
import dlt
from pyspark.sql.functions import to_timestamp, count, sum, col

# --- SILVER LAYER: Cleaning & Deduplication ---

@dlt.table(
  name="events_cleaned", 
  comment="Cleaned ecommerce behavior events"
)
@dlt.expect_or_drop("valid_timestamp", "event_time IS NOT NULL") # DQ Constraint 
@dlt.expect_or_drop("valid_price", "price > 0") # DQ Constraint 

def sample_users_streaming_pipeline():
    return (dlt.read_stream("workspace.dev_bronze_layer.events_raw") 
            .withColumn("event_time", to_timestamp(col("event_time")))
            .dropDuplicates(["user_id", "event_time", "product_id"]))

# --- GOLD LAYER: Aggregated Metrics ---
@dlt.table(
    name="customer_metrics",
    comment="Business Deliverable 1: Customer Journey Analysis"
)
def customer_metrics():
    return (
        dlt.read("events_cleaned")
        .groupBy("user_id")
        .agg({"product_id": "count", "price": "sum"})
        .withColumnRenamed("count(product_id)", "total_interactions")
        .withColumnRenamed("sum(price)", "total_spend")
    )

@dlt.table(
    name="product_performance",
    comment="Business Deliverable 2: Product Sales Analysis"
)

def product_performance():
    # Reading from the silver table defined in the same pipeline
    return (
        dlt.read("events_cleaned")
        .filter(col("event_type") == "purchase")
        .groupBy("product_id", "category_code", "brand")
        .agg(
            count("event_type").alias("total_purchases"),
            sum("price").alias("total_revenue")
        )
    )